### Imports

In [ ]:
import os
os.chdir('..')

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from modules.data_setup import create_specific_dataset, sports_dataframe, class_distribution
from modules.training import run_training
from modules.architecture import Conv, Conv2
from modules.utils import nb_parameters

In [ ]:
seed = 42 #DON'T CHANGE THIS VALUE

### Preparations of dataset and subsets: 

We will work with gray and then with colored data to compare the impact of color and also to reduce de number of parametters by 3 for the gray images. Our hypothesis is that for sport comparation, color will be usefull so improve accuracy.

Functions used can be found in `data_setup.py`.

#### 3 fairly different classes

In [ ]:
DIFF_CLASSES_3 = ["football", "nascar racing", "olympic wrestling"]
print(f"For DIFF_CLASSES_3: Loading {len(DIFF_CLASSES_3)} classes: {DIFF_CLASSES_3}...")
train_ds_Gray_DIFF_3, val_ds_Gray_DIFF_3, test_ds_Gray_DIFF_3 = create_specific_dataset(DIFF_CLASSES_3, "Gray")
train_ds_Color_DIFF_3, val_ds_Color_DIFF_3, test_ds_Color_DIFF_3 = create_specific_dataset(DIFF_CLASSES_3, "Color")

In [ ]:
class_distribution(sports_dataframe, DIFF_CLASSES_3, save_path=f"DIFF_CLASSES_3_distribution.png")

#### 3 fairly similar classes

In [ ]:
SIM_CLASSES_3 = ["football", "rugby", "ampute football"]
print(f"For SIM_CLASSES_3: Loading {len(SIM_CLASSES_3)} classes: {SIM_CLASSES_3}...")
train_ds_Gray_SIM_3, val_ds_Gray_SIM_3, test_ds_Gray_SIM_3 = create_specific_dataset(SIM_CLASSES_3, "Gray")
train_ds_Color_SIM_3, val_ds_Color_SIM_3, test_ds_Color_SIM_3 = create_specific_dataset(SIM_CLASSES_3, "Color")

In [ ]:
class_distribution(sports_dataframe, SIM_CLASSES_3, save_path=f"SIM_CLASSES_3_distribution.png")

#### 6 fairly different classes

In [ ]:
DIFF_CLASSES_6 = ["football", "nascar racing", "olympic wrestling", "surfing", "archery", "arm wrestling"]
print(f"For DIFF_CLASSES_6: Loading {len(DIFF_CLASSES_6)} classes: {DIFF_CLASSES_6}...")
train_ds_Gray_DIFF_6, val_ds_Gray_DIFF_6, test_ds_Gray_DIFF_6 = create_specific_dataset(DIFF_CLASSES_6, "Gray")
train_ds_Color_DIFF_6, val_ds_Color_DIFF_6, test_ds_Color_DIFF_6 = create_specific_dataset(DIFF_CLASSES_6, "Color")

In [ ]:
class_distribution(sports_dataframe, DIFF_CLASSES_6, save_path=f"DIFF_CLASSES_6_distribution.png")

#### 12 classes

In [ ]:
CLASSES_12 = ["football", "olympic wrestling", "archery", "arm wrestling", "swimming", "ski jumping", "rowing", "pole dancing", "formula 1 racing", "boxing", "baton twirling", "axe throwing"]
print(f"For CLASSES_12: Loading {len(CLASSES_12)} classes: {CLASSES_12[:5]}...")
train_ds_Gray_12, val_ds_Gray_12, test_ds_Gray_12 = create_specific_dataset(CLASSES_12, "Gray")
train_ds_Color_12, val_ds_Color_12, test_ds_Color_12 = create_specific_dataset(CLASSES_12, "Color")
train_ds_Gray_12_64, val_ds_Gray_12_64, test_ds_Gray_12_64 = create_specific_dataset(CLASSES_12, "Gray", 64)
train_ds_Color_12_64, val_ds_Color_12_64, test_ds_Color_12_64 = create_specific_dataset(CLASSES_12, "Color", 64)

In [ ]:
class_distribution(sports_dataframe, CLASSES_12, save_path=f"CLASSES_12_distribution.png")

#### 20 classes

In [ ]:
CLASSES_20 = ["football", "nascar racing", "olympic wrestling", "surfing", "archery", "arm wrestling", "swimming", "ampute football", "skydiving", "sky surfing", "ski jumping", "rowing", "pommel horse", "pole dancing", "pole climbing", "formula 1 racing", "boxing", "baton twirling", "axe throwing", "water cycling"]
print(f"For CLASSES_20: Loading {len(CLASSES_20)} classes: {CLASSES_20[:5]}...")
train_ds_Gray_20, val_ds_Gray_20, test_ds_Gray_20 = create_specific_dataset(CLASSES_20, "Gray")
train_ds_Color_20, val_ds_Color_20, test_ds_Color_20 = create_specific_dataset(CLASSES_20, "Color")

In [ ]:
class_distribution(sports_dataframe, CLASSES_20, save_path=f"CLASSES_20_distribution.png")

#### 100 classes (whole dataset)

In [ ]:
CLASSES_100 = sports_dataframe['labels'].unique().tolist()
print(f"For CLASSES_100: Loading {len(CLASSES_100)} classes: {CLASSES_100[:5]}...")
train_ds_Gray_100, val_ds_Gray_100, test_ds_Gray_100 = create_specific_dataset(CLASSES_100, "Gray")
train_ds_Color_100, val_ds_Color_100, test_ds_Color_100 = create_specific_dataset(CLASSES_100, "Color")

In [ ]:
class_distribution(sports_dataframe, CLASSES_100, save_path=f"CLASSES_100_distribution.png")

### Before we start

- Model Net with activation function, regularization and convolution. We set convolution by default to None, if so, it just flat the image. This + `class Conv` can be found in `modules\architecture.py`

- Optimizers, schedulers and other small functions can be found in `modules\utils.py`

- Finally, the `train` and `run_training` functions can be found in `modules\training.py`

### Experiments

#### Step 1: Best hidden layer configuration

Let's start our process with default parameters. For the activation function we will use "relu". We will not use any regularizations, not any schedulers and for the optimizer we will use SGD by default for now.

For the convolution, we use first no convolution at all even we now this is a bad idea. It mean we will just flat the images and give it to the model.

Let's arbitrarily set the learning rate to 0.001 and the number of epochs to 20 for now. Our first objective is to find the best hiddens layers configurations to use it for the rest of the process.
Let's first work with the set of 20 sport (with somes differents and some similar)

Note : In the idea, our results should be the worst at the beginning of the process and the best at the end.

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_12c_256h128h, net_Gray_12c_256h128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[256, 128], epochs=20, lr=0.001, seed=seed, pr=plot_results)
metrics_Gray_12c_128h64h, net_Gray_12c_128h64h= run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001, seed=seed, pr=plot_results)
metrics_Gray_12c_128h, net_Gray_12c_128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128], epochs=20, lr=0.001, seed=seed, pr=plot_results)
metrics_Gray_12c_512h_128h, net_Gray_12c_512h_128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[512, 128], epochs=20, lr=0.001, seed=seed, pr=plot_results)
metrics_Gray_12c_64h_32h, net_Gray_12c_64h_32h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[64, 32], epochs=20, lr=0.001, seed=seed, pr=plot_results)
metrics_Gray_12c_128h_64h_32h, net_Gray_12c_128h_64h_32h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64, 32], epochs=20, lr=0.001, seed=seed, pr=plot_results)

Comparing differents results and not focucing on overfitting of ran, best results are gets with x hiddens layers of 256 and 128 neurons which represent 12 880 788 parameters. But we choosed to use 128 and 64 which is a bit less efficient but represent the half of parameters. It will be easier then, so we will use it for the next of the process.

Let's add now a 2D convolution. As we don't know what is the best configuration (well of course we have theorical ideas but we need to check in practice), we will try with differents kernel sizes (3 and 7). We also need to determine the number of output channel of the convolution which is the number of filter of the convolution. We will try 3 arbitral values : 8, 16 and 32. It represent 6 ran.

#### Step 2: Adding a 2D convolution

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_12c_8o_3k_poolmax, net_Gray_12c_8o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_16o_3k_poolmax, net_Gray_12c_16o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=16, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_32o_3k_poolmax, net_Gray_12c_32o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=32, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_8o_7k_poolmax, net_Gray_12c_8o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=7, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_16o_7k_poolmax, net_Gray_12c_16o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=16, kernel_size=7, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_32o_7k_poolmax, net_Gray_12c_32o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=32, kernel_size=7, pooling="max", seed=seed, pr=plot_results)

Changing the kernel size does not affect the results and passing from 8 to 16 filters improve them but using 32 filters does not change a lot. But since the number of parameters explode with the number of filter, we'll keep working with 8 filters.

Let's compare it with average pooling.

In [ ]:
metrics_Gray_12c_8o_3k_poolaverage, net_Gray_12c_8o_3k_poolaverage = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="average", seed=seed, pr=plot_results)

As expected, poolmax is better than poolaverage.

Until here, we used relu because we know by theoritical that this is the best but we should compare the results between the two others activations functions

#### Step 3: Activation functions

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_12_tanh, net_Gray_12c_tanh = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], activation_function="tanh", epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_sigmoid, net_Gray_12c_sigmoid = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], activation_function="sigmoid", epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

#### Step 4: Computation Loss

Here as we see, relu still be the best option. Now, let's focus on the loss computation to check model behaviour

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_12c, net_Gray_12c = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

#### Step 5: Optimisers and Scheduler

We see that the model is overfitting. We then need to add some regularization to prevent our modele to to so. But before, Let's add optimisers and scheduler.

We also should also add better optimizer than SGD : Adam to get a better weight computation and schedulers to have the best possible learning rate : step that divided learning rate by 2 every 10 epochs.

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_12c_optim, net_Gray_12c_weight_optim = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12,  hidden_sizes=[128, 64], optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

#### Step 6: Regularization

We observe that the loss is more stable because of the scheduler. We can now focus on regularisation. Let's try with weight decay and changing the penalty

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.0001, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.001, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.01, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

Not any value of weigh decay fix the overfitting. We even had underfitting with to high values. We may try with an other regularisation, let's use dropout

In [ ]:
metrics_Gray_12c_dropout, net_Gray_12c_dropout = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="dropout", reg_param=0.1, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

metrics_Gray_12c_dropout, net_Gray_12c_dropout = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="dropout", reg_param=0.3, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


Dropout did not solved overfitting. An other idea we had is to work with a different sizes for the images. Let's try with 64 instead of 224.

In [ ]:
metrics_Gray_12c_64, net_Gray_12c_64 = run_training(train_ds_Gray_12_64, val_ds_Gray_12_64, num_classes=12, hidden_sizes=[128, 64], optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

We are still blocked here. We add an other idea, adding a convolution layer wich will reduce parameter number and our idea is to try again regularisation on this new model

In [ ]:
metrics_Gray_12c, net_Gray_12c = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


Results are really better but overfitting is high, let's try dropout again

In [ ]:
metrics_Gray_12c_dropout, net_Gray_12c_dropout = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="dropout", reg_param=0.1, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


We did not solved overfitting. Finally, we choose to juste add early stopping to keep the model at the best moment

In [ ]:
metrics_Gray_12c_early, net_Gray_12c_early = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=12, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


#### Step 7: Adding Color

Overfitting is now not fixed but at least limited. Now, let's run our model with color. In theory, accurate results should be better. We also want to check if the model do not overfit.

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Color_12c, net_Color_12c = run_training(train_ds_Color_12, val_ds_Color_12, num_classes=12, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=3, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

#### Step 8: Comparation

Our hypothesis is verified : Color help the model to be more accurate.

Now, our model is more stable and our parameters seems to be relevants.

We are finally at the step we wanted to. Now, we will run and compare results between 3 similar sport, 3 completly different (where we expect having the most accurate results), 6 differents classes, 20 classes and finally 100 sports. Our objectif here is to get conclusion on how the increase number of classes and the difficulty of the classifications affect a model training. We will work with grey and colors datasets

In [ ]:
# Select if you want to plot results or not
plot_results = False

metrics_Gray_3c_similar, net_Gray_3c_similar = run_training(train_ds_Gray_SIM_3, val_ds_Gray_SIM_3, num_classes=3, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Color_3c_similar, net_Color_3c_similar = run_training(train_ds_Color_SIM_3, val_ds_Color_SIM_3, num_classes=3, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=3, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Gray_3c_diff, net_Gray_3c_diff = run_training(train_ds_Gray_DIFF_3, val_ds_Gray_DIFF_3, num_classes=3, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Color_3c_diff, net_Color_3c_diff = run_training(train_ds_Color_DIFF_3, val_ds_Color_DIFF_3, num_classes=3, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=3, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Gray_6c, net_Gray_6c = run_training(train_ds_Gray_DIFF_6, val_ds_Gray_DIFF_6, num_classes=6, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Color_6c, net_Color_6c = run_training(train_ds_Color_DIFF_6, val_ds_Color_DIFF_6, num_classes=6, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=3, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Gray_20c, net_Gray_20c = run_training(train_ds_Gray_20, val_ds_Gray_20, num_classes=20, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Color_20c, net_Color_20c = run_training(train_ds_Color_20, val_ds_Color_20, num_classes=20, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=3, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Gray_100c, net_Gray_100c = run_training(train_ds_Gray_100, val_ds_Gray_100, num_classes=100, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=1, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)


metrics_Color_100c, net_Color_100c = run_training(train_ds_Color_100, val_ds_Color_100, num_classes=100, hidden_sizes=[128, 64], regularization="early_stopping", optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv2, in_channels=3, out_channels=8, kernel_size=3, pooling="max", seed=seed, pr=plot_results)

